In [1]:
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
from collections import defaultdict

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("\nHunting for the correct dataset...")
path_1 = "train.parquet"
path_2 = "netflix-prize-data/train.parquet"
correct_path = None

if os.path.exists(path_1) and os.path.getsize(path_1) / 1e6 > 500:
    correct_path = path_1
elif os.path.exists(path_2) and os.path.getsize(path_2) / 1e6 > 500:
    correct_path = path_2

if not correct_path:
    raise FileNotFoundError("Could not find the full 548 MB train.parquet file.")

print(f"Loading data from: {correct_path}")
df_train = pd.read_parquet(correct_path)
df_test = pd.read_parquet(correct_path.replace("train.parquet", "test.parquet"))

Using device: cuda

Hunting for the correct dataset...
Loading data from: netflix-prize-data/train.parquet


In [3]:
print("Filtering for active users...")
user_counts = df_train.groupby('user_id').size()
active_users = user_counts[user_counts >= 50].index

df_train_active = df_train[df_train['user_id'].isin(active_users)]
df_train_sub = df_train_active.sample(n=2000000, random_state=42)

# Free memory
del df_train, df_train_active
gc.collect()

# Create index mappings for PyTorch Embeddings (IDs must be 0 to N-1)
unique_users = pd.concat([df_train_sub['user_id'], df_test['user_id']]).unique()
unique_movies = pd.concat([df_train_sub['movie_id'], df_test['movie_id']]).unique()

user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
movie_to_idx = {mid: idx for idx, mid in enumerate(unique_movies)}

df_train_sub['user_idx'] = df_train_sub['user_id'].map(user_to_idx)
df_train_sub['movie_idx'] = df_train_sub['movie_id'].map(movie_to_idx)

# Test set might have cold-start users/movies not in our sample. We drop them to evaluate fairly.
df_test = df_test[df_test['user_id'].isin(user_to_idx) & df_test['movie_id'].isin(movie_to_idx)].copy()
df_test['user_idx'] = df_test['user_id'].map(user_to_idx)
df_test['movie_idx'] = df_test['movie_id'].map(movie_to_idx)

num_users = len(unique_users)
num_movies = len(unique_movies)
print(f"Mapped {num_users:,} unique users and {num_movies:,} unique movies.")

Filtering for active users...
Mapped 470,592 unique users and 17,698 unique movies.


In [4]:
class NetflixDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.movies = torch.tensor(df['movie_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        
    def __len__(self):
        return len(self.ratings)
    
    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]

class NCF(nn.Module):
    def __init__(self, num_users, num_movies, embed_dim=32):
        super(NCF, self).__init__()
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.movie_embed = nn.Embedding(num_movies, embed_dim)
        
        # Deep Neural Network layers
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
        
    def forward(self, user, movie):
        u = self.user_embed(user)
        m = self.movie_embed(movie)
        x = torch.cat([u, m], dim=-1)
        return self.mlp(x).squeeze(-1)

# Initialize Loaders
train_loader = DataLoader(NetflixDataset(df_train_sub), batch_size=2048, shuffle=True)
test_loader = DataLoader(NetflixDataset(df_test), batch_size=4096, shuffle=False)

In [5]:
model = NCF(num_users, num_movies, embed_dim=32).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5
print("\nStarting NCF Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    t0 = time.time()
    
    for users, movies, ratings in train_loader:
        users, movies, ratings = users.to(device), movies.to(device), ratings.to(device)
        
        optimizer.zero_grad()
        preds = model(users, movies)
        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss (MSE): {total_loss/len(train_loader):.4f} | Time: {time.time()-t0:.1f}s")


Starting NCF Training...
Epoch 1/5 | Loss (MSE): 1.7753 | Time: 18.4s
Epoch 2/5 | Loss (MSE): 1.2711 | Time: 17.9s
Epoch 3/5 | Loss (MSE): 1.1898 | Time: 18.0s
Epoch 4/5 | Loss (MSE): 1.1402 | Time: 17.8s
Epoch 5/5 | Loss (MSE): 1.0839 | Time: 18.0s


In [6]:
print("\nEvaluating NCF on Test Set...")
model.eval()
test_preds = []
test_actuals = []
test_uids = []
test_iids = []

with torch.no_grad():
    for users, movies, ratings in test_loader:
        u_batch, m_batch = users.to(device), movies.to(device)
        preds = model(u_batch, m_batch)
        
        test_preds.extend(preds.cpu().numpy())
        test_actuals.extend(ratings.numpy())
        test_uids.extend(users.numpy())
        test_iids.extend(movies.numpy())

final_ncf_rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))

# Calculate MAP@10
top_n = defaultdict(list)
for uid, iid, est, true_r in zip(test_uids, test_iids, test_preds, test_actuals):
    top_n[uid].append((iid, est, true_r))

for uid, user_ratings in top_n.items():
    user_ratings.sort(key=lambda x: x[1], reverse=True)
    top_n[uid] = user_ratings[:10]

aps = []
for uid, user_ratings in top_n.items():
    hits, sum_precs = 0, 0
    for i, (iid, est, true_r) in enumerate(user_ratings):
        if true_r >= 3.5:
            hits += 1
            sum_precs += hits / (i + 1.0)
    aps.append(sum_precs / min(len(user_ratings), 10) if hits > 0 else 0.0)

ncf_map10 = np.mean(aps)

print("\n=======================================================")
print("  PHASE 3 COMPLETE — DEEP LEARNING RESULTS")
print("=======================================================")
print(f"  Final NCF RMSE   : {final_ncf_rmse:.4f}")
print(f"  Final NCF MAP@10 : {ncf_map10:.4f} ({ncf_map10 * 100:.2f}%)")
print("=======================================================")


Evaluating NCF on Test Set...

  PHASE 3 COMPLETE — DEEP LEARNING RESULTS
  Final NCF RMSE   : 1.0723
  Final NCF MAP@10 : 0.5560 (55.60%)


In [7]:
import pandas as pd
import random

# 1. Load Movie Titles
print("Loading movie metadata...")
df_movies = pd.read_parquet("movies.parquet")
movie_title_map = dict(zip(df_movies['movie_id'], df_movies['title']))

# 2. Package Phase 3 Predictions into a DataFrame
# (Using the lists generated at the end of your Phase 3 evaluation loop)
results_df = pd.DataFrame({
    'user_idx': test_uids,
    'movie_idx': test_iids,
    'predicted_rating': test_preds,
    'actual_rating': test_actuals
})

# Map back to original IDs if you want, but for title lookup we need the original movie_id.
# Let's reverse the dictionary we made in Phase 3
idx_to_movie = {v: k for k, v in movie_to_idx.items()}
results_df['movie_id'] = results_df['movie_idx'].map(idx_to_movie)
results_df['title'] = results_df['movie_id'].map(movie_title_map)

# 3. Select a Random Active User to Analyze
sample_user = random.choice(results_df['user_idx'].unique())
user_results = results_df[results_df['user_idx'] == sample_user].copy()

# 4. Generate Top-10 Recommendations (What the model thinks are the best)
top_10 = user_results.sort_values(by='predicted_rating', ascending=False).head(10)

print(f"\n=======================================================")
print(f" TOP-10 RECOMMENDATIONS FOR USER [ID: {sample_user}]")
print(f"=======================================================")
for i, row in enumerate(top_10.itertuples(), 1):
    print(f"{i:2d}. {row.title[:45]:<45} | Pred: {row.predicted_rating:.2f} | Actual: {row.actual_rating:.1f}")

# 5. Success vs. Failure Analysis
print("\n=======================================================")
print(" SUCCESS & FAILURE ANALYSIS")
print("=======================================================")

# Success: We predicted >= 3.5, and they actually rated it >= 3.5
successes = user_results[(user_results['predicted_rating'] >= 3.5) & (user_results['actual_rating'] >= 3.5)]
print(f"✅ SUCCESS CASES (Recommended correctly): {len(successes)}")
for row in successes.head(3).itertuples():
    print(f"   - {row.title[:40]:<40} (Pred: {row.predicted_rating:.2f}, Actual: {row.actual_rating:.1f})")

# Failure: We predicted >= 3.5, but they hated it (<= 2.5)
failures = user_results[(user_results['predicted_rating'] >= 3.5) & (user_results['actual_rating'] <= 2.5)]
print(f"\n❌ FAILURE CASES (Bad recommendations): {len(failures)}")
for row in failures.head(3).itertuples():
    print(f"   - {row.title[:40]:<40} (Pred: {row.predicted_rating:.2f}, Actual: {row.actual_rating:.1f})")

# Missed Opportunities: We predicted < 3.5, but they loved it (>= 4.0)
misses = user_results[(user_results['predicted_rating'] < 3.5) & (user_results['actual_rating'] >= 4.0)]
print(f"\n⚠️ MISSED OPPORTUNITIES (Underestimated): {len(misses)}")
for row in misses.head(3).itertuples():
    print(f"   - {row.title[:40]:<40} (Pred: {row.predicted_rating:.2f}, Actual: {row.actual_rating:.1f})")
print("=======================================================")

Loading movie metadata...

 TOP-10 RECOMMENDATIONS FOR USER [ID: 197874]
 1. Crash                                         | Pred: 3.69 | Actual: 5.0
 2. The Longest Yard                              | Pred: 3.35 | Actual: 2.0
 3. The Hitchhiker's Guide to the Galaxy          | Pred: 2.88 | Actual: 2.0

 SUCCESS & FAILURE ANALYSIS
✅ SUCCESS CASES (Recommended correctly): 1
   - Crash                                    (Pred: 3.69, Actual: 5.0)

❌ FAILURE CASES (Bad recommendations): 0

⚠️ MISSED OPPORTUNITIES (Underestimated): 0
